In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import seaborn as sns

from sklearn.base import BaseEstimator , TransformerMixin
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import cross_validate
from sklearn.compose import ColumnTransformer
from imblearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from imblearn.over_sampling import SMOTE
from dataclasses import dataclass
import joblib
from sklearn.svm import SVC
from sklearn.model_selection import RandomizedSearchCV
import os
from sklearn.metrics import roc_auc_score

import scipy

In [ ]:
linux_path = r"/run/media/drdrakken/Elements/Sonstiges/Programmieren/Machine Learning/csvs/driving/porto-seguro-safe-driver-prediction/train.csv"
df = pd.read_csv(linux_path)

In [ ]:
@dataclass
class Config:
    target: str = "target"
    seed: int = 1234
    test_size: float = 0.2
    cross_iterations: int = 5
    save_path : str = r"/home/drdrakken/Downloads/ModelSaveDir/save_driver-ptk"
config = Config()

In [ ]:
class DataClass():
    def __init__(self , DataFrame = df):
        self.x = DataFrame.drop([config.target] , axis = 1)
        self.y = DataFrame[config.target]

        self.numerical_data = self.x.select_dtypes(include = np.number).columns
        self.categorical_data = self.x.select_dtypes(exclude = np.number).columns
data = DataClass()            

In [ ]:
class Visualize():

    def __init__(self):
        self.data = data.x

    def iqr(self, TargetCol):
        q1 = self.data[TargetCol].quantile(0.25)
        q3 = self.data[TargetCol].quantile(0.75)
        iqr = q3 - q1
        return q1 , q3 , iqr
    
    def skew(self , SkewCol):
        data_skew = self.data[SkewCol].skew()
        print(f"Skewness of Col:{SkewCol}->:{data_skew}")

    def plot(self):
        for vis in self.data.numerical_data:
            fig , axes = plt.subplots(3 , 1 , figsize = (10 , 10) , dpi = 200)
            q1 , q3 , iqr = self.iqr(TargetCol = vis)
            mean = self.data[vis].mean()
            self.skew(SkewCol = vis)

            sns.histplot(data = self.data , x = vis , ax = axes[0])
            axes[0].axvline(q1 , color = "green")
            axes[0].axvline(q3 , color = "red")
            axes[0].axvline(mean , color = "yellow")
            axes[0].set_title(f"Histplot for:{vis}")

            sns.boxplot(data = self.data , x = vis , ax = axes[1])
            axes[1].axvline(q1 , color = "green")
            axes[1].axvline(q3 , color = "red")
            axes[1].axvline(mean , color = "yellow")
            axes[1].set_title(f"Boxplot for:{vis}")

            sns.scatterplot(data = self.data , x = vis , ax = axes[2])
            axes[2].axvline(q1 , color = "green")
            axes[2].axvline(q3 , color = "red")
            axes[2].axvline(mean , color = "yellow")
            axes[2].set_title(f"Scatterplot for:{vis}")

            plt.tight_layout()
            plt.show()


In [ ]:
class DataPreprcocess(BaseEstimator, TransformerMixin):

    def fit(self , X , y = None):
        numerical_cols = X.select_dtypes(include = np.number).columns
        categorical_cols = X.select_dtypes(exclude = np.number).columns

        self.preprocess = ColumnTransformer([
            ("numerical_data_process" , Pipeline([
                ("imputer" , SimpleImputer(strategy = "mean")),
                ("scale" , MinMaxScaler()),
            ]),numerical_cols),

            ("categorical_data_process" , Pipeline([
                ("imputer" , SimpleImputer(strategy = "most_frequent")),
                ("encoder" , OneHotEncoder(handle_unknown = "ignore" , sparse_output=False)),
            ]),categorical_cols),
        ])
        self.preprocess.fit(X)
        return self
    
    def transform(self , X , y = None):
        return self.preprocess.transform(X)

In [ ]:
class DataTransform(BaseEstimator , TransformerMixin):

    def fit(self , X , y = None):
        return self
    
    def transform(self, X , y = None):
        return self.new_features(X)
    
    def new_features(self , X , y = None):
        x = X.copy()

        return x
    
    def combine_features(self , X):
        x = X.copy()
        #combined = [comb for comb in x.columns if comb startswith("ps_ind")]
        return x

In [ ]:
def model_varianz():
    return {

        "LogisticRegression":LogisticRegression(random_state = config.seed , max_iter = 1000),
        "SVC":SVC(random_state = config.seed , kernel = "linear" , probability = True ),
        "RandomForestClassifier":RandomForestClassifier(random_state = config.seed),
        "DecisionTreeClassifier":DecisionTreeClassifier(random_state = config.seed),
        
    }

In [ ]:
def custom_grid_search(estimator , parameters , X_train , y_train):
    grid = RandomizedSearchCV(estimator= estimator , 
                              return_train_score= True,
                              param_distributions = parameters,
                              random_state = config.seed, 
                              cv = config.cross_iterations,
                              n_iter = 20,
                              scoring = "roc_auc")
    
    grid.fit(X_train , y_train)
    return grid

In [ ]:
def custom_model_parameters():
    return {

        "LogisticRegression": {
            "estimator__C": scipy.stats.loguniform(1e-4, 1e2),
            "estimator__penalty": ["l2"],
            "estimator__solver": ["lbfgs"],
        },

        "DecisionTreeClassifier": {
            "estimator__criterion": ["gini", "entropy"],
            "estimator__max_depth": [None, 5, 10, 20],
            "estimator__min_samples_split": scipy.stats.randint(2, 20),
            "estimator__min_samples_leaf": scipy.stats.randint(1, 10),
        },

        "RandomForestClassifier": {
            "estimator__n_estimators": scipy.stats.randint(100, 500),
            "estimator__max_depth": scipy.stats.randint(5, 40),
            "estimator__min_samples_split": scipy.stats.randint(2, 20),
            "estimator__min_samples_leaf": scipy.stats.randint(1, 10),
            "estimator__max_features": ["sqrt", "log2"],
            "estimator__bootstrap": [True, False],
        },

        "LinearSVC": {
            "estimator__C": scipy.stats.loguniform(1e-4, 1e2),
            "estimator__loss": ["hinge", "squared_hinge"],
            "estimator__dual": [True],
            "estimator__max_iter": [5000, 10000],
        },

        "XGBClassifier": {
            "estimator__n_estimators": scipy.stats.randint(100, 500),
            "estimator__learning_rate": scipy.stats.loguniform(1e-3, 0.3),
            "estimator__max_depth": scipy.stats.randint(3, 10),
            "estimator__subsample": scipy.stats.uniform(0.6, 0.4),
            "estimator__colsample_bytree": scipy.stats.uniform(0.6, 0.4),
            "estimator__gamma": scipy.stats.uniform(0, 5),
            "estimator__min_child_weight": scipy.stats.randint(1, 10),
        },

    }


In [ ]:
def data_split():
    X_train , X_test , y_train , y_test = train_test_split(data.x,
                                                           data.y,
                                                           shuffle = True,
                                                           random_state = config.seed,
                                                           test_size = config.test_size,
                                                           stratify = data.y)
    
    return  X_train , X_test , y_train , y_test

In [ ]:
def custom_data_validation(estimator , X_train , y_train):
    kfold = StratifiedKFold(n_splits = config.cross_iterations , shuffle = True , random_state = config.seed)

    return cross_validate(estimator = estimator,
                          X= X_train,
                          y= y_train,
                          return_train_score = True,
                          scoring="roc_auc",
                          cv = kfold,
                          return_estimator=True,
                          verbose=config.verbose,
                          )

In [ ]:
def custom_pipeline(estimator , transform = False , smote = False):
    steps = []

    if transform:
        steps.append(("TransformPeformance" , DataTransform()))

    steps.append(("BasePeformance" , DataPreprcocess()))
    if smote:
        steps.append(("smote" , SMOTE()))
        
    steps.append(("estimator" , estimator))
    return Pipeline(steps)

In [ ]:
class Benchmark():

    def __init__(self):
        self.X_train , self.X_test , self.y_train , self.y_test = data_split()
        self.results = []
        self.train()

    def train(self):

        for estimator_name , estimators in model_varianz().items():

            base_pipe = custom_pipeline(estimator=estimators , transform= False , smote=False)
            transformed_pipe = custom_pipeline(estimator=estimators , transform= True , smote=True)

        if self.use_cv:
            base_cv = custom_data_validation(estimator=base_pipe,
                                             X_train=self.X_train,
                                             y_train=self.y_train)
            
            transformed_cv = custom_data_validation(estimator=transformed_pipe,
                                             X_train=self.X_train,
                                             y_train=self.y_train)
            print(f"keys;{base_cv.keys()}")
            self.results.append({

                "base_cv_train_performance":base_cv["train_score"].mean(),
                "base_cv_train_performance":base_cv["test_score"].mean(),
                "base_cv_std_performance":base_cv["train_score"].std(),

                "transformed_cv_train_performance":transformed_cv["train_score"].mean(),
                "transformed_cv_train_performance":transformed_cv["train_score"].mean(),
                "transformed_cv_std_performance":transformed_cv["train_score"].std(),
            })

        if self.use_grid_search:
            base_grid = custom_grid_search(estimator=base_pipe,
                                           X_train=self.X_train,
                                           y_train=self.y_train,
                                           parameters=custom_model_parameters()[estimator_name])

            transformed_grid = custom_grid_search(estimator=base_pipe,
                                           X_train=self.X_train,
                                           y_train=self.y_train,
                                           parameters=custom_model_parameters()[estimator_name])
        self.results.append({

            "base_cv_train_performance":base_grid.best_estimator,
            "transformed_cv_train_performance":transformed_grid.best_estimator,
        })


In [ ]:
Benchmark()

Train:{estimator_name}


KeyError: 'Base_test_performance'